In [ ]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer


pairs = [
    ("hi", "salut"),
    ("how are you", "comment allez vous"),
    ("i am fine", "je vais bien"),
    ("thank you", "merci"),
    ("good morning", "bonjour"),
    ("good night", "bonne nuit")
]

# Add SOS/EOS manually
src_texts = [p[0] for p in pairs]
tgt_texts = ["<sos> " + p[1] + " <eos>" for p in pairs]


def make_tokenizer():
    # KEEP < > symbols
    return Tokenizer(filters='')

eng_tok = make_tokenizer()
fr_tok  = make_tokenizer()

eng_tok.fit_on_texts(src_texts)
fr_tok.fit_on_texts(tgt_texts)

eng_sequences = eng_tok.texts_to_sequences(src_texts)
fr_sequences  = fr_tok.texts_to_sequences(tgt_texts)

max_eng = max(len(s) for s in eng_sequences)
max_fr  = max(len(s) for s in fr_sequences)

eng_pad = pad_sequences(eng_sequences, maxlen=max_eng, padding='post')
fr_pad  = pad_sequences(fr_sequences, maxlen=max_fr, padding='post')

# Decoder target (shift left)
fr_target = np.zeros_like(fr_pad)
fr_target[:, :-1] = fr_pad[:, 1:]

eng_vocab = len(eng_tok.word_index) + 1
fr_vocab  = len(fr_tok.word_index) + 1


latent_dim = 128

# Encoder
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(eng_vocab, latent_dim)(encoder_inputs)
enc_out, state_h, state_c = LSTM(latent_dim, return_state=True)(enc_emb)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(None,))
dec_emb_layer = Embedding(fr_vocab, latent_dim)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
dec_out, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(fr_vocab, activation='softmax')
output = decoder_dense(dec_out)

model = Model([encoder_inputs, decoder_inputs], output)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")

model.summary()


model.fit([eng_pad, fr_pad], fr_target, epochs=200, verbose=0)
print("Training complete!")


encoder_model = Model(encoder_inputs, encoder_states)

# Decoder inference model
dec_state_input_h = Input(shape=(latent_dim,))
dec_state_input_c = Input(shape=(latent_dim,))
dec_states_input = [dec_state_input_h, dec_state_input_c]

dec_emb2 = dec_emb_layer(decoder_inputs)
dec_out2, h2, c2 = decoder_lstm(dec_emb2, initial_state=dec_states_input)
dec_out2 = decoder_dense(dec_out2)

decoder_model = Model(
    [decoder_inputs] + dec_states_input,
    [dec_out2, h2, c2]
)


index_to_fr = {i: w for w, i in fr_tok.word_index.items()}

sos_token = fr_tok.word_index["<sos>"]
eos_token = fr_tok.word_index["<eos>"]

def translate(sentence, max_len=30):
    seq = eng_tok.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_eng, padding='post')

    states = encoder_model.predict(seq, verbose=0)

    target_seq = np.array([[sos_token]])
    decoded = []

    for _ in range(max_len):
        preds, h, c = decoder_model.predict([target_seq] + states, verbose=0)
        next_id = np.argmax(preds[0, -1, :])

        if next_id == eos_token:
            break

        decoded.append(index_to_fr.get(next_id, ""))

        target_seq = np.array([[next_id]])
        states = [h, c]

    return " ".join(decoded)


tests = [
    "hi",
    "thank you",
    "good morning",
    "how are you",
    "i am fine"
]

for t in tests:
    print(t, "→", translate(t))